In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv(r"C:\Users\drisy\Downloads\warehouse_order_processing.csv")

print("=" * 60)
print("STEP 1: BASIC INFO")
print("=" * 60)
print(f"Shape: {df.shape}")
print("\nData Types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())

# ─────────────────────────────────────────────
# 2. MISSING VALUE CHECK
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: MISSING VALUES")
print("=" * 60)
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# If there were missing values, you'd handle them like:
# df.fillna(df.mean(), inplace=True)  # for numerical columns

# ─────────────────────────────────────────────
# 3. DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: DESCRIPTIVE STATISTICS")
print("=" * 60)
print(df.describe().round(2))

# ─────────────────────────────────────────────
# 4. DUPLICATE CHECK
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: DUPLICATE ROWS")
print("=" * 60)
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"Duplicates removed. New shape: {df.shape}")

# ─────────────────────────────────────────────
# 5. OUTLIER DETECTION (IQR Method)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: OUTLIER DETECTION (IQR Method)")
print("=" * 60)

def detect_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower) | (data[col] > upper)]
    return len(outliers), lower, upper

for col in df.columns:
    count, low, high = detect_outliers_iqr(df, col)
    print(f"{col:35s} → {count:4d} outliers  [bounds: {low:.2f}, {high:.2f}]")

# Visualize outliers with boxplots
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].boxplot(df[col])
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel("Value")
plt.suptitle("Boxplots for Outlier Detection", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("boxplots_outliers.png", dpi=150)
plt.close()
print("\nBoxplot saved as 'boxplots_outliers.png'")

# ─────────────────────────────────────────────
# 6. OUTLIER TREATMENT (Capping / Winsorization)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: OUTLIER TREATMENT (Capping at IQR bounds)")
print("=" * 60)

df_clean = df.copy()
for col in df_clean.columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)])
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
    print(f"{col:35s} → {before} outliers capped")

# ─────────────────────────────────────────────
# 7. DISTRIBUTION / NORMALITY CHECK
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: NORMALITY CHECK (Shapiro-Wilk on sample of 200)")
print("=" * 60)

for col in df_clean.columns:
    sample = df_clean[col].sample(200, random_state=42)
    stat, p = stats.shapiro(sample)
    result = "Normal ✓" if p > 0.05 else "Not Normal ✗"
    print(f"{col:35s} → p={p:.4f}  {result}")

# Distribution plots
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(df_clean.columns):
    axes[i].hist(df_clean[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")
plt.suptitle("Distributions of All Variables", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("distributions.png", dpi=150)
plt.close()
print("\nDistribution plot saved as 'distributions.png'")

# ─────────────────────────────────────────────
# 8. CORRELATION MATRIX
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: CORRELATION MATRIX")
print("=" * 60)
corr = df_clean.corr().round(3)
print(corr)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True,
            linewidths=0.5, annot_kws={"size": 9})
plt.title("Correlation Heatmap", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.close()
print("Correlation heatmap saved as 'correlation_heatmap.png'")

# ─────────────────────────────────────────────
# 9. MULTICOLLINEARITY CHECK (VIF)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: MULTICOLLINEARITY CHECK (VIF)")
print("=" * 60)

from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df_clean.drop(columns=["Order_Processing_Time_min"])
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data["Status"] = vif_data["VIF"].apply(
    lambda v: "OK ✓" if v < 5 else ("Moderate ⚠" if v < 10 else "High ✗")
)
print(vif_data.to_string(index=False))
print("\nRule: VIF < 5 → OK | 5–10 → Moderate | >10 → High multicollinearity")

# ─────────────────────────────────────────────
# 10. FEATURE SCALING (Standardization)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 10: FEATURE SCALING (StandardScaler)")
print("=" * 60)

X = df_clean.drop(columns=["Order_Processing_Time_min"])
y = df_clean["Order_Processing_Time_min"]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Before scaling (mean | std):")
print(X.agg(['mean', 'std']).round(2))
print("\nAfter scaling (mean | std):")
print(X_scaled.agg(['mean', 'std']).round(2))

# ─────────────────────────────────────────────
# 11. TRAIN-TEST SPLIT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 11: TRAIN-TEST SPLIT (80% train / 20% test)")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f"Training set   : X={X_train.shape}, y={y_train.shape}")
print(f"Test set       : X={X_test.shape}, y={y_test.shape}")

# ─────────────────────────────────────────────
# 12. SAVE PREPROCESSED DATA
# ─────────────────────────────────────────────
train_df = X_train.copy()
train_df["Order_Processing_Time_min"] = y_train.values
test_df = X_test.copy()
test_df["Order_Processing_Time_min"] = y_test.values

train_df.to_csv("warehouse_train.csv", index=False)
test_df.to_csv("warehouse_test.csv", index=False)

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE!")
print("=" * 60)
print("Files saved:")
print("  → warehouse_train.csv  (800 rows)")
print("  → warehouse_test.csv   (200 rows)")
print("  → boxplots_outliers.png")
print("  → distributions.png")
print("  → correlation_heatmap.png")

STEP 1: BASIC INFO
Shape: (1000, 9)

Data Types:
 Num_Items_Ordered              int64
Warehouse_Distance_m         float64
Num_Workers_Available          int64
Order_Weight_kg              float64
Hour_of_Day                    int64
Num_Pending_Orders             int64
Packing_Complexity_Score       int64
Equipment_Downtime_min       float64
Order_Processing_Time_min    float64
dtype: object

First 5 rows:
    Num_Items_Ordered  Warehouse_Distance_m  Num_Workers_Available  \
0                 39                212.12                      7   
1                 29                299.20                     14   
2                 15                270.02                      9   
3                 43                177.04                     15   
4                  8                276.04                      3   

   Order_Weight_kg  Hour_of_Day  Num_Pending_Orders  Packing_Complexity_Score  \
0            14.71           16                  50                        10   
1         